# 从零实现 SynthID-Text：在 DeepSeek 上嵌入水印并用 Weighted Mean 检测

1、用一个小词表精确证明：显式 Tournament 等价于修改 logits

2、实现极简 SynthID logits processor，并与 Hugging Face 逐元素对比

3、从零实现 repeated-context mask 与 weighted mean 检测

4、分析全词表版本的时间、显存复杂度与优化方向

5、用 DeepSeek 生成文本，并评估不同长度下的检测效果

这里实现的是公开参考方案，不是 Gemini 线上服务的私有密钥或哈希配置。Weighted Mean 是无需训练的检测器；
Google 的开源仓库还提供了效果更强、但需要水印/非水印训练数据的 Bayesian detector。

论文：https://www.nature.com/articles/s41586-024-08025-4

Google 参考实现：https://github.com/google-deepmind/synthid-text

注：完整生成实验建议在 A100 上运行。





In [ ]:
# Colab 已预装 PyTorch、NumPy、Pandas、SciPy、scikit-learn 和 Matplotlib。
%pip -q install "transformers==5.14.1" sacreble





In [ ]:
import json
import math
from pathlib import Path
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from sacrebleu.metrics import CHRF
from scipy.stats import gaussian_kde, norm
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    SynthIDTextWatermarkLogitsProcessor,
    SynthIDTextWatermarkingConfig,
    set_seed,
)

CORPUS_URL = "https://raw.githubusercontent.com/GenTang/GenTang.github.io/main/content/zh/blog/watermarking_on_aigc/data/kgw_corpus.jsonl"
SHOWCASE_URL = "https://raw.githubusercontent.com/GenTang/GenTang.github.io/main/content/zh/blog/watermarking_on_aigc/data/kgw_showcase.jsonl"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Hugging Face 文档中的公开示例配置：9 个 key 对应 9 层 Tournament。
KEYS = [654, 400, 836, 123, 340, 443, 597, 160, 57]
NGRAM_LEN = 5
CONTEXT_HISTORY_SIZE = 1024
SAMPLING_TABLE_SEED = 0
SAMPLING_TABLE_SIZE = 2**16

# Google 官方 weighted mean 的默认权重从 10 线性下降到 1。
WEIGHT_START, WEIGHT_END = 10.0, 1.0

# 分别检测前 32、64、128 和 256 个“未被重复上下文掩码排除”的有效位置。
PREFIXES = [32, 64, 128, 256]
TARGET_FPR = 0.01
THEORETICAL_Z_THRESHOLD = float(norm.ppf(1 - TARGET_FPR))
SEED = 1024

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = (
    (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    if DEVICE.type == "cuda"
    else torch.float32
)




## 1、先证明：显式 Tournament 等价于修改 logits

单层 Tournament 从原分布 $p$ 独立抽两个候选 $a,b$。若二者的二值分数 $g$ 不同，让 $g=1$ 的候选获胜；
若相同则让第一个候选获胜。下面不做 Monte Carlo 近似，而是枚举小词表中的全部 $(a,b)$，精确计算获胜分布。

令 $q=\sum_v p(v)g(v)$，同一个分布也可以直接写为

$$p'(v)=p(v)\left(1+g(v)-q\right).$$

因此，“显式抽两个 token 比赛”与“把 logits 改成 $\log p'$ 后正常采样”完全等价。先验证单层，
再连续应用 3 层不同的 g-values：每层都等价，所以组合后的多层 Tournament 也等价。





In [ ]:
def explicit_tournament(prob: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """枚举两个候选的全部 V² 个有序组合，返回显式 Tournament 的精确分布。"""
    tokens = torch.arange(len(prob))

    # 这三个矩阵操作就是双重 for 循环的向量化写法：
    # for first in tokens:
    #     for second in tokens:
    #         output[winner(first, second)] += p(first) * p(second)
    first = tokens[:, None].expand(-1, len(prob))
    second = tokens[None, :].expand(len(prob), -1)
    winner = torch.where(g[first] >= g[second], first, second)  # g 平局时第一个获胜
    joint_prob = prob[:, None] * prob[None, :]
    return torch.zeros_like(prob).scatter_add_(0, winner.flatten(), joint_prob.flatten())


def closed_form_tournament(prob: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """不枚举候选对，直接返回与单层 Tournament 等价的修改后分布。"""
    q = (prob * g).sum()
    return prob * (1 + g - q)


prob = torch.tensor([0.10, 0.20, 0.30, 0.40], dtype=torch.float64)
g_layers = torch.tensor([
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 0, 1, 1],
])

# 第一项验证单层；随后逐层重复，验证 3 层组合后的最终分布。
explicit_one = explicit_tournament(prob, g_layers[0])
closed_one = closed_form_tournament(prob, g_layers[0])
torch.testing.assert_close(explicit_one, closed_one)

explicit_multi, closed_multi = prob.clone(), prob.clone()
for layer_g in g_layers:
    explicit_multi = explicit_tournament(explicit_multi, layer_g)
    closed_multi = closed_form_tournament(closed_multi, layer_g)
torch.testing.assert_close(explicit_multi, closed_multi)

# 模型实际接收 logits；softmax(log(p')) 会恢复同一个 Tournament 分布。
torch.testing.assert_close(explicit_multi, torch.softmax(closed_multi.log(), dim=-1))
display(pd.DataFrame({
    "token": torch.arange(len(prob)),
    "original_p": prob,
    "one_layer_explicit": explicit_one,
    "one_layer_modified_logits": closed_one,
    "three_layer_explicit": explicit_multi,
    "three_layer_modified_logits": closed_multi,
}))





## 2、极简的 SynthID logits processor

下面只保留 Hugging Face 精确对齐所不可省略的四部分：LCG 哈希、固定二值表、逐层闭式更新和重复 context
跳过。真正的水印更新在 `update_logits` 中只有 7 行；`MinimalSynthID` 其余代码只是把它接入
`model.generate()` 并保存最近 context。




In [ ]:
MULTIPLIER = 6_364_136_223_846_793_005


def hash_int64(current: torch.LongTensor, values: torch.LongTensor) -> torch.LongTensor:
    """与 Hugging Face 相同的 int64 LCG 累积哈希。

    参数
    ----
    current : (...,) int64 tensor
        每条数据的初始哈希；前置形状必须能与 values[..., 0] 广播。
    values : (..., N) int64 tensor
        依次吸收到哈希中的 N 个整数，例如 context token、候选 token 或 layer key。

    返回
    ----
    (...,) int64 tensor
        吸收全部 values 后的哈希。int64 自然溢出是算法的一部分。
    """
    for value in values.unbind(dim=-1):
        current = (current + value) * MULTIPLIER + 1
    return current


def candidate_g(
    context: torch.LongTensor,
    candidates: torch.LongTensor,
    keys: torch.LongTensor,
    table: torch.LongTensor,
) -> tuple[torch.LongTensor, torch.LongTensor]:
    """计算当前 context 下全部候选 token、全部 Tournament 层的 g-values。

    参数
    ----
    context : (B, H) int64 tensor
        每个 batch 样本最近 H=ngram_len-1 个 token。
    candidates : (B, V) int64 tensor
        每个样本的 V 个候选 token ID；全词表版本通常是 0...V-1。
    keys : (m,) int64 tensor
        m 层 Tournament 各自的秘密整数 key。
    table : (S,) int64 tensor
        由 sampling_table_seed 固定生成的 0/1 伪随机表。

    返回
    ----
    g_values : (B, V, m) int64 tensor
        每个候选 token 在每个 Tournament layer 上的二值分数。
    context_hash : (B,) int64 tensor
        只包含 context 的哈希，用于识别重复 context。
    """
    # hash(context) -> hash(context, candidate) -> hash(context, candidate, layer_key)
    initial = torch.ones(context.shape[0], device=context.device, dtype=torch.long)
    context_hash = hash_int64(initial, context)
    hashes = hash_int64(context_hash[:, None], candidates[..., None])
    hashes = hash_int64(hashes[..., None], keys[None, None, :, None])

    # 哈希对表长取模，确定性地取得 Bernoulli(0.5) 的 0/1 g-value。
    return table[hashes % table.numel()], context_hash


def update_logits(
    logits: torch.FloatTensor,
    g_values: torch.LongTensor,
) -> torch.FloatTensor:
    """把 m 层显式 Tournament 转成等价的 logits 更新。

    参数
    ----
    logits : (B, V) floating tensor
        语言模型对 V 个候选 token 给出的原始 logits。
    g_values : (B, V, m) 0/1 tensor
        candidate_g 生成的 m 层 Tournament 分数。

    返回
    ----
    (B, V) floating tensor
        修改后的 log-probabilities；从它采样等价于依次进行 m 层显式 Tournament。
    """
    probs = logits.softmax(dim=-1)
    for layer in range(g_values.shape[-1]):
        g = g_values[..., layer].to(probs.dtype)
        g_mass = (probs * g).sum(dim=-1, keepdim=True)  # q_l = Σ_v p(v)g_l(v)
        probs *= 1 + g - g_mass                         # p'(v) = p(v)(1+g_l(v)-q_l)
    log_probs = probs.log()
    return torch.where(torch.isfinite(log_probs), log_probs, torch.finfo(log_probs.dtype).min)


class MinimalSynthID(LogitsProcessor):
    """与 Hugging Face 输出一致、但只保留必要逻辑的 SynthID processor。"""

    def __init__(self, ngram_len, keys, sampling_table_size, sampling_table_seed,
                 context_history_size, device):
        """初始化极简 SynthID processor。

        参数
        ----
        ngram_len : int
            一个 g-value 使用的 n-gram 长度；其中前 n-1 个 token 是 context，最后一个是候选 token。
        keys : sequence[int]
            Tournament layer keys；key 数量就是水印深度 m。
        sampling_table_size : int
            固定 0/1 伪随机表的长度 S。越大越不容易发生表索引碰撞，但占用更多常驻显存。
        sampling_table_seed : int
            生成伪随机表的公开复现实验 seed；嵌入端和检测端必须相同。
        context_history_size : int
            保存多少个最近 context 哈希；context 重复时跳过水印，以满足单序列非失真条件。
        device : torch.device | str
            哈希、采样表和 logits 所在设备，例如 "cuda:0" 或 "cpu"。
        """
        self.ngram_len, self.device = ngram_len, torch.device(device)
        self.keys = torch.tensor(keys, device=device)

        # 相同 seed + 相同 device 会生成与 Hugging Face 完全相同的 0/1 表。
        rng = torch.Generator(device=device).manual_seed(sampling_table_seed)
        self.table = torch.randint(0, 2, (sampling_table_size,), generator=rng, device=device)
        self.history_size = context_history_size

        # context/history 属于一次 generate 调用，新的文本必须使用新的 processor 实例。
        self.context = self.history = None

    @property
    def depth(self):
        return len(self.keys)

    @torch.no_grad()
    def __call__(
        self,
        input_ids: torch.LongTensor,
        logits: torch.FloatTensor,
    ) -> torch.FloatTensor:
        """修改当前生成步的 logits。

        参数
        ----
        input_ids : (B, L) int64 tensor
            model.generate 当前已经拥有的完整 token 序列。初始化后只需读取最后一个新 token 更新 context。
        logits : (B, V) floating tensor
            当前生成步尚未加 SynthID 水印的 logits。

        返回
        ----
        (B, V) floating tensor
            首次出现的 context 返回 Tournament 修改后的 logits；重复 context 原样返回输入 logits。
        """
        batch, vocab_size = logits.shape

        # 1. Hugging Face 用 H 个 0 初始化 context；后续每步左移并追加刚生成的 token。
        if self.context is None:
            self.context = torch.zeros(batch, self.ngram_len - 1, dtype=torch.long, device=self.device)
            self.history = torch.zeros(batch, self.history_size, dtype=torch.long, device=self.device)
        else:
            self.context = torch.cat((self.context, input_ids[:, -1:]), dim=-1)[:, 1:]

        # 2. 为全词表 V 个候选和 m 层 key 计算形状 (B,V,m) 的 g-values。
        candidates = torch.arange(vocab_size, device=self.device)[None].expand(batch, -1)
        g_values, context_hash = candidate_g(self.context, candidates, self.keys, self.table)

        # 3. 连续执行 m 次闭式 Tournament 分布更新。
        watermarked = update_logits(logits, g_values)

        # 4. 已使用过的 context 跳过水印；无论是否重复，都把本次 context 放入 history。
        context_hash = context_hash[:, None]
        repeated = (self.history == context_hash).any(dim=-1, keepdim=True)
        self.history = torch.cat((context_hash, self.history), dim=-1)[:, :-1]
        return torch.where(repeated, logits, watermarked)


def sequence_g_values(
    input_ids: torch.LongTensor,
    processor: MinimalSynthID,
) -> torch.LongTensor:
    """检测端一次重建整段序列的 g-values。

    参数
    ----
    input_ids : (B, L) int64 tensor
        待检测的完整 token 序列，不包含 prompt。
    processor : MinimalSynthID
        提供 ngram_len、keys 和 sampling table；必须与嵌入端配置相同。

    返回
    ----
    (B, L-ngram_len+1, m) int64 tensor
        每个完整 n-gram 在 m 个 Tournament layers 上的 g-values。
    """
    ngrams = input_ids.unfold(1, processor.ngram_len, 1)
    hashes = hash_int64(
        torch.ones(ngrams.shape[:2], dtype=torch.long, device=input_ids.device), ngrams
    )
    hashes = hash_int64(hashes[..., None], processor.keys[None, None, :, None])
    return processor.table[hashes % processor.table.numel()]


def repetition_mask(
    input_ids: torch.LongTensor,
    processor: MinimalSynthID,
) -> torch.BoolTensor:
    """检测端屏蔽第二次及以后出现的相同 context。

    参数
    ----
    input_ids : (B, L) int64 tensor
        与 sequence_g_values 相同的待检测 token 序列。
    processor : MinimalSynthID
        提供 ngram_len 和 context_history_size。

    返回
    ----
    (B, L-ngram_len+1) bool tensor
        首次出现的 context 为 True，history 中已出现的 context 为 False。
    """
    contexts = input_ids[:, :-1].unfold(1, processor.ngram_len - 1, 1)
    history = torch.zeros(
        input_ids.shape[0], processor.history_size, dtype=torch.long, device=input_ids.device
    )
    mask = []
    for context in contexts.unbind(dim=1):
        context_hash = hash_int64(
            torch.ones(input_ids.shape[0], dtype=torch.long, device=input_ids.device), context
        )[:, None]
        mask.append(~(history == context_hash).any(dim=-1, keepdim=True))
        history = torch.cat((context_hash, history), dim=-1)[:, :-1]
    return torch.cat(mask, dim=-1)





## 3、Weighted Mean 水印检测

检测不需要再次运行语言模型，只需 tokenizer、密钥和相同的哈希配置。对未被重复 context mask 排除的
$T$ 个位置及 $m$ 层 g-values，Google 的 weighted mean 为

$$S=\frac{1}{T\sum_l w_l}\sum_{t=1}^{T}\sum_{l=1}^{m}w_l g_{t,l}.$$

越靠前的 Tournament 层信号越强，所以官方默认让 $w_l$ 从 10 线性下降到 1。零假设（没有该水印）下，
$g_{t,l}\sim\mathrm{Bernoulli}(0.5)$，因此 $\mathbb E[S]=0.5$。若近似认为各项独立，

$$\mathrm{Var}(S)=\frac{0.25\sum_l w_l^2}{T(\sum_l w_l)^2}.$$

原始 weighted mean 才是核心检测分数；下面的 z-score 只负责按文本长度标准化，便于使用同一个理论阈值。
公开实现也建议：若直接使用 raw score 比较不同长度文本，应为每个长度单独校准阈值。





In [ ]:
class SynthIDWeightedMeanDetector:
    """从 token IDs 重建 g-values、应用重复 context mask，并计算 weighted mean。"""

    def __init__(
        self,
        processor: MinimalSynthID,
        weights: Sequence[float] | torch.Tensor | None = None,
    ) -> None:
        self.processor = processor
        if weights is None:
            weights = torch.linspace(
                WEIGHT_START, WEIGHT_END, processor.depth, device=processor.device
            )
        self.weights = torch.as_tensor(
            weights, dtype=torch.float64, device=processor.device
        )
        if self.weights.shape != (processor.depth,):
            raise ValueError(f"weights 必须有 {processor.depth} 个元素")
        if (self.weights < 0).any() or self.weights.sum() <= 0:
            raise ValueError("weights 必须非负且总和大于 0")

    def weighted_mean(
        self, g_values: torch.Tensor, mask: torch.Tensor
    ) -> torch.Tensor:
        """批量计算官方定义的 weighted mean；结果形状为 (B,)。"""
        if g_values.ndim != 3 or mask.shape != g_values.shape[:2]:
            raise ValueError("g_values 应为 (B,T,D)，mask 应为 (B,T)")
        weights = self.weights.to(g_values.device)
        masked = g_values.to(torch.float64) * mask[..., None].to(torch.float64)
        count = mask.sum(dim=1)
        return (masked * weights).sum(dim=(1, 2)) / (count * weights.sum())

    def null_std(self, tokens: int) -> float:
        """Bernoulli(0.5) 独立近似下 weighted mean 的标准差。"""
        weights = self.weights
        variance = 0.25 * float((weights.square().sum() / weights.sum().square())) / tokens
        return math.sqrt(variance)

    def raw_threshold(self, tokens: int, z_threshold: float) -> float:
        """把统一 z 阈值转换为指定有效长度下的 raw weighted-mean 阈值。"""
        return 0.5 + z_threshold * self.null_std(tokens)

    def __call__(
        self,
        token_ids: Sequence[int] | torch.LongTensor,
        max_scored_tokens: int | None = None,
    ) -> dict | None:
        ids = torch.as_tensor(
            token_ids, dtype=torch.long, device=self.processor.device
        ).flatten()
        if ids.numel() < self.processor.ngram_len:
            return None

        batch = ids[None, :]
        g_values = sequence_g_values(batch, self.processor)
        mask = repetition_mask(batch, self.processor)
        valid_g = g_values[0, mask[0]]
        if max_scored_tokens is not None:
            valid_g = valid_g[:max_scored_tokens]
        tokens = int(valid_g.shape[0])
        if tokens == 0:
            return None

        # 所有保留下来的位置 mask 都为 1；仍调用批量函数保证只有一份公式实现。
        valid_mask = torch.ones(1, tokens, dtype=torch.bool, device=ids.device)
        score = float(self.weighted_mean(valid_g[None, :], valid_mask)[0])
        std = self.null_std(tokens)
        z = (score - 0.5) / std
        return {
            "tokens": tokens,
            "weighted_mean": score,
            "null_std": std,
            "z": z,
            "p_normal": float(norm.sf(z)),
            "layer_means": valid_g.to(torch.float32).mean(dim=0).cpu().tolist(),
        }





## 4、与 Hugging Face 官方实现逐元素对比

给极简版与官方 processor 输入完全相同的 logits，并连续模拟生成调用。最后一次调用故意复用先前的
4-token context，因此哈希、g-values、逐层概率更新、生成状态和 repeated-context 跳过只要有一处不同，
逐元素断言就不会通过。





In [ ]:
CONFIG = dict(
    ngram_len=NGRAM_LEN,
    keys=KEYS,
    sampling_table_size=SAMPLING_TABLE_SIZE,
    sampling_table_seed=SAMPLING_TABLE_SEED,
    context_history_size=CONTEXT_HISTORY_SIZE,
)

# 对比只需要同样的词表索引，不需要下载语言模型。
TEST_DEVICE, TEST_VOCAB_SIZE = DEVICE, 4096
scratch = MinimalSynthID(**CONFIG, device=TEST_DEVICE)
official_config = SynthIDTextWatermarkingConfig(**CONFIG)
official = SynthIDTextWatermarkLogitsProcessor(
    **official_config.to_dict(), device=TEST_DEVICE
)
assert torch.equal(scratch.table, official.sampling_table)

# 第 9 次调用时 context [33, 44, 55, 66] 第二次出现，双方都应返回原始 logits。
running_ids = torch.tensor([[11, 22]], device=TEST_DEVICE)
generator = torch.Generator(device=TEST_DEVICE).manual_seed(SEED)
for next_id in [33, 44, 55, 66, 33, 44, 55, 66, 99]:
    scores = torch.randn(1, TEST_VOCAB_SIZE, generator=generator, device=TEST_DEVICE)
    torch.testing.assert_close(
        scratch(running_ids, scores), official(running_ids, scores), rtol=1e-6, atol=1e-6
    )
    running_ids = torch.cat(
        (running_ids, torch.tensor([[next_id]], device=TEST_DEVICE)), dim=1
    )

# weighted mean 与公开公式的直接 tensor 表达式一致。
detector_processor = MinimalSynthID(**CONFIG, device=TEST_DEVICE)
detector = SynthIDWeightedMeanDetector(detector_processor)
test_g = torch.randint(0, 2, (2, 7, len(KEYS)), device=TEST_DEVICE)
test_mask = torch.tensor(
    [[1, 1, 0, 1, 1, 0, 1], [1, 0, 1, 1, 0, 1, 1]],
    dtype=torch.bool,
    device=TEST_DEVICE,
)
weights = torch.linspace(10, 1, len(KEYS), dtype=torch.float64, device=TEST_DEVICE)
reference_score = (
    (test_g.to(torch.float64) * test_mask[..., None] * weights).sum(dim=(1, 2))
    / (test_mask.sum(dim=1) * weights.sum())
)
torch.testing.assert_close(detector.weighted_mean(test_g, test_mask), reference_score)
print("极简 logits processor 与 Hugging Face 逐元素一致；Weighted Mean 公式验证通过")





## 5、性能分析

“极简”只表示代码短，不会自动降低算法的计算量。这个与 Hugging Face 精确对齐的版本在每个生成步都要为
全词表计算 `vocab_size × depth` 个 g-values，时间复杂度为 $O(BVm)$，并对概率分布做 $m$ 次乘法与归约。

对本实验的 DeepSeek 模型，$V=151{,}936$、$m=9$、batch size 为 1：每步产生 1,367,424 个 g-values。
单个 int64 张量约 10.4 MiB；哈希索引和 g-values 同时存在时仅这两项就约 20.9 MiB。显存通常不是主要问题，
额外的显存带宽、哈希运算和 9 次全词表归约才可能影响 tokens/s。

大模型的 forward pass 通常仍占主要时间，但这里是 1.5B 小模型且词表很大，水印开销占比可能比 7B/70B
模型更明显，不能仅因为代码只有几十行就断言“几乎没有下降”。论文报告的生产系统使用经过优化的
30 层实现：Gemma 7B-IT 在 TPU 上从 15.527 ms/token 增至 15.615 ms/token（约 0.57%）；这个数字不能直接
外推到 Python、A100、DeepSeek 1.5B 和全词表实现。

若性能优先，可以只对 top-k 候选计算 g-values，复杂度降为 $O(Bkm)$；Google 的参考实现也提供这种路径。
但候选截断顺序必须与生成配置固定，否则就不再与 Hugging Face 当前的全词表 processor 逐元素一致。





In [ ]:
def is_watermarked(z_score: float, threshold: float = THEORETICAL_Z_THRESHOLD) -> bool:
    """判断长度标准化后的 weighted-mean z-score 是否超过阈值。"""
    return z_score > threshold


def detect_ids(
    ids: Sequence[int],
    threshold: float = THEORETICAL_Z_THRESHOLD,
    max_scored_tokens: int | None = None,
) -> dict:
    """检测 token 序列；max_scored_tokens 用于固定有效 context 数。"""
    result = detector(ids, max_scored_tokens=max_scored_tokens)
    if result is None:
        return {"detected": False, "reason": "文本太短或没有可评分 context"}
    return {
        **result,
        "raw_threshold": detector.raw_threshold(result["tokens"], threshold),
        "z_threshold": threshold,
        "detected": is_watermarked(result["z"], threshold),
    }


def detect_text(text: str, threshold: float = THEORETICAL_Z_THRESHOLD) -> dict:
    """将普通文本分词后检测公开配置对应的 SynthID-Text 水印。"""
    ids = tokenizer(text, add_special_tokens=False).input_ids
    return detect_ids(ids, threshold)


display(pd.DataFrame({
    "scored_tokens": PREFIXES,
    "raw_score_threshold_at_1%_FPR": [
        detector.raw_threshold(length, THEORETICAL_Z_THRESHOLD) for length in PREFIXES
    ],
}))





## 6、生成复述文本

与 KGW 实验保持一致：使用更快的 DeepSeek-R1-Distill-Qwen-1.5B，将 `<think>` 在 prompt 端关闭；
若模型再次进入思考，只保留最后一个 `</think>` 之后的原始生成 token。每次生成都新建 logits processor，
避免把上一段文本的 context history 带进下一段。





In [ ]:
# 从这里开始才下载实验语料、tokenizer 和模型。
data = pd.read_json(CORPUS_URL, lines=True)
showcase_original = pd.read_json(SHOWCASE_URL, lines=True).text.iloc[0]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, attn_implementation="sdpa"
).to(DEVICE).eval()
MODEL_DEVICE, VOCAB_SIZE = next(model.parameters()).device, model.config.vocab_size
print("语料段落数：", len(data), "运行设备：", MODEL_DEVICE, "词表大小：", VOCAB_SIZE)

THINK_OPEN_IDS = tokenizer("<think>", add_special_tokens=False).input_ids
THINK_CLOSE_IDS = tokenizer("</think>", add_special_tokens=False).input_ids
THINK_PREFILL_IDS = tokenizer("</think>\n", add_special_tokens=False).input_ids


def prompt_inputs(text: str):
    """构造 R1 对话输入，并在 prompt 端关闭思考区。"""
    chars = len("".join(text.split()))
    messages = [{"role": "user", "content": (
        "请忠实复述下面的文本，保留事实、逻辑关系和结论，但使用你自己的表达。"
        f"复述正文长度尽量与原文相同（原文约{chars}个非空白字符，允许上下浮动10%）。"
        "不要分析或输出前言，只输出复述正文。\n\n原文：\n" + text
    )}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    close = torch.tensor([THINK_PREFILL_IDS])
    inputs["input_ids"] = torch.cat([inputs["input_ids"], close], dim=1)
    inputs["attention_mask"] = torch.cat(
        [inputs["attention_mask"], torch.ones_like(close)], dim=1
    )
    return inputs.to(MODEL_DEVICE)


def find_last(ids: list[int], pattern: list[int]) -> int:
    """返回最后一个完整 pattern 的起点；不存在则返回 -1。"""
    return next(
        (index for index in range(len(ids) - len(pattern), -1, -1)
         if ids[index:index + len(pattern)] == pattern),
        -1,
    )


def body_tokens(ids: list[int]) -> tuple[list[int], bool]:
    """若 R1 再次思考，只保留最后一个 </think> 之后的正文 token。"""
    close = find_last(ids, THINK_CLOSE_IDS)
    if close >= 0:
        return ids[close + len(THINK_CLOSE_IDS):], True
    return ([], False) if find_last(ids, THINK_OPEN_IDS) >= 0 else (ids, True)


@torch.inference_mode()
def generate(text: str, watermarked: bool, seed: int = SEED) -> dict:
    """生成无水印或 9 层 SynthID-Text 水印复述，并保留原始 token IDs。"""
    set_seed(seed)
    inputs = prompt_inputs(text)
    source_tokens = len(tokenizer(text, add_special_tokens=False).input_ids)
    kwargs = dict(
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        max_new_tokens=min(512, max(128, math.ceil(source_tokens * 1.15))),
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if watermarked:
        kwargs["logits_processor"] = LogitsProcessorList([
            MinimalSynthID(**CONFIG, device=MODEL_DEVICE)
        ])

    output = model.generate(**inputs, **kwargs)[0, inputs["input_ids"].shape[1]:].tolist()
    stop_ids = {tokenizer.eos_token_id, tokenizer.pad_token_id}
    output = output[:next(
        (index for index, token in enumerate(output) if token in stop_ids), len(output)
    )]
    output, valid = body_tokens(output)
    text_out = tokenizer.decode(output, skip_special_tokens=True).strip()
    return {
        "watermarked": watermarked,
        "seed": seed,
        "text": text_out,
        "token_ids": output,
        "valid": valid and bool(text_out),
    }





## 7、直观展示、g-values 与文字指标

同一份原文分别生成无水印和 SynthID-Text 水印版本。展示文本后，查看 weighted mean、各层 g-value
均值和 token×layer 热力图。水印文本应在前层出现更多的 1，而无水印文本各层均值应在 0.5 附近。





In [ ]:
showcase = [generate(showcase_original, mode) for mode in (False, True)]
if not all(item["valid"] for item in showcase):
    raise ValueError("showcase 中存在未完成思考或空正文，请更换 SEED 后重试")

display(Markdown("## 原文\n" + showcase_original))
for item in showcase:
    mode = "watermarked" if item["watermarked"] else "unwatermarked"
    display(Markdown(f"## {mode}\n" + item["text"]))





In [ ]:
showcase_detection = [{"mode": "original", **detect_text(showcase_original)}]
showcase_detection += [
    {"mode": "watermarked" if item["watermarked"] else "unwatermarked",
     **detect_ids(item["token_ids"])}
    for item in showcase
]
display(pd.DataFrame(showcase_detection).drop(columns="layer_means", errors="ignore"))





In [ ]:
def valid_g_values(ids: Sequence[int]) -> torch.Tensor:
    """返回去掉重复 context 后的 (T, depth) g-value 矩阵。"""
    batch = torch.tensor([list(ids)], dtype=torch.long, device=MODEL_DEVICE)
    g_values = sequence_g_values(batch, detector_processor)
    mask = repetition_mask(batch, detector_processor)
    return g_values[0, mask[0]].cpu()


fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for axis, item in zip(axes, showcase):
    g = valid_g_values(item["token_ids"])
    shown = g[:min(128, len(g))].T
    axis.imshow(shown, aspect="auto", interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    axis.set(
        title="watermarked" if item["watermarked"] else "unwatermarked",
        xlabel="有效 token 位置（最多展示 128）",
        ylabel="Tournament layer",
        yticks=range(len(KEYS)),
    )
plt.show()

layer_comparison = pd.DataFrame({
    "layer": np.arange(1, len(KEYS) + 1),
    "weight": detector.weights.cpu().numpy(),
    "unwatermarked_g_mean": valid_g_values(showcase[0]["token_ids"]).float().mean(0).numpy(),
    "watermarked_g_mean": valid_g_values(showcase[1]["token_ids"]).float().mean(0).numpy(),
})
display(layer_comparison.round(3))





In [ ]:
chrf = CHRF()
bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-zh-v1.5")
bge = AutoModel.from_pretrained("BAAI/bge-small-zh-v1.5").to(MODEL_DEVICE).eval()


@torch.inference_mode()
def embeddings(texts: list[str], batch_size: int = 32) -> torch.Tensor:
    """分批计算归一化 BGE 句向量，返回 CPU tensor。"""
    result = []
    for start in range(0, len(texts), batch_size):
        batch = bge_tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=512, return_tensors="pt",
        ).to(MODEL_DEVICE)
        result.append(F.normalize(bge(**batch).last_hidden_state[:, 0], dim=-1).cpu())
    return torch.cat(result)


def similarity(reference: str | list[str], candidate: str | list[str]) -> pd.DataFrame:
    """计算 chrF、语义余弦相似度和 token 长度比。"""
    references = [reference] if isinstance(reference, str) else reference
    candidates = [candidate] if isinstance(candidate, str) else candidate
    if len(references) == 1:
        references = references * len(candidates)
    reference_emb, candidate_emb = embeddings(references), embeddings(candidates)
    reference_ids = tokenizer(references, add_special_tokens=False).input_ids
    candidate_ids = tokenizer(candidates, add_special_tokens=False).input_ids
    return pd.DataFrame({
        "chrF": [
            chrf.sentence_score(candidate_text, [reference_text]).score / 100
            for reference_text, candidate_text in zip(references, candidates)
        ],
        "Semantic cosine": (reference_emb * candidate_emb).sum(dim=1).numpy(),
        "Length ratio": [
            len(candidate_tokens) / len(reference_tokens)
            for reference_tokens, candidate_tokens in zip(reference_ids, candidate_ids)
        ],
    })


unwatermarked_showcase, watermarked_showcase = showcase
comparisons = [
    ("Unwatermarked vs original", showcase_original, unwatermarked_showcase["text"]),
    ("Watermarked vs unwatermarked", unwatermarked_showcase["text"], watermarked_showcase["text"]),
]
for title, reference, candidate in comparisons:
    display(Markdown(f"### {title}"), similarity(reference, candidate))





## 8、主实验

全部语料分别生成无水印和水印版本。结果逐条写入 JSONL，Colab 中断后可以继续。检测阈值预先固定为
Bernoulli(0.5) 独立近似对应的 1% 单侧阈值，不使用测试数据校准；最后报告实际 FPR、TPR 与 ROC-AUC。





In [ ]:
# 调试时可设为 20；None 表示使用全部语料。
MAX_SAMPLES = None
sources = data.sample(frac=1, random_state=SEED).reset_index(drop=True)
if MAX_SAMPLES:
    sources = sources.head(MAX_SAMPLES).copy()
print("实验段落数：", len(sources))

# 修改模型、语料、prompt、密钥或算法参数时必须更换文件名。
RESULTS = Path("/content/synthid_weighted_mean_r1_1_5b_generations_v1.jsonl")
records = [json.loads(line) for line in RESULTS.open(encoding="utf-8")] if RESULTS.exists() else []
done = {
    (int(row["source_id"]), bool(row["watermarked"]))
    for row in records if row.get("valid", False)
}
tasks = [
    (source, mode)
    for source in sources.itertuples(index=False)
    for mode in (False, True)
]
completed = sum((int(source.id), mode) in done for source, mode in tasks)
print(f"总任务数：{len(tasks)}，已完成：{completed}，待生成：{len(tasks) - completed}")

with RESULTS.open("a", encoding="utf-8") as output:
    for source, mode in tasks:
        task = (int(source.id), mode)
        if task in done:
            continue
        row = {
            "source_id": int(source.id),
            "source_text": source.text,
            **generate(source.text, mode, SEED),
        }
        output.write(json.dumps(row, ensure_ascii=False) + "\n")
        output.flush()
        records.append(row)
        done.add(task)
        completed += 1
        if completed % 10 == 0:
            print(f"[{completed}/{len(tasks)}] 已保存", flush=True)

valid_records = [row for row in records if row.get("valid", False)]
generated = pd.DataFrame(valid_records)
print(f"有效生成结果：{len(valid_records)}，无效结果：{len(records) - len(valid_records)}")





In [ ]:
unwatermarked = generated[~generated["watermarked"]][["source_id", "text"]]
watermarked = generated[generated["watermarked"]][["source_id", "text"]]
pairs = unwatermarked.merge(
    watermarked, on="source_id", suffixes=("_unwatermarked", "_watermarked")
)

# 以同一原文的 unwatermarked 输出为基准，衡量水印额外带来的变化。
quality = similarity(
    pairs["text_unwatermarked"].tolist(), pairs["text_watermarked"].tolist()
)
display(quality.agg(["mean", "std"]).round(3))





## 9、不同文本长度下的 Weighted Mean 分布与检测效果

每段文本按前 32/64/128/256 个有效 context 评分，保证长度可比。无水印的 raw score 应围绕 0.5，
标准化后的 z-score 应近似 $N(0,1)$；有水印分布则随累计证据增加而右移。





In [ ]:
def prefix_scores(ids: Sequence[int], source_id: int, mode: str) -> list[dict]:
    """为同一文本计算多个固定有效长度下的 weighted-mean 分数。"""
    full = detector(ids)
    if full is None:
        return []
    rows = []
    for length in PREFIXES:
        if full["tokens"] >= length:
            result = detector(ids, max_scored_tokens=length)
            rows.append({
                "source_id": source_id,
                "mode": mode,
                "length": length,
                **result,
            })
    return rows


samples = [
    (source.id, "original", tokenizer(source.text, add_special_tokens=False).input_ids)
    for source in sources.itertuples(index=False)
]
samples += [
    (row["source_id"], "watermarked" if row["watermarked"] else "unwatermarked",
     row["token_ids"])
    for row in valid_records
]
score_rows = [
    score
    for source_id, mode, ids in samples
    for score in prefix_scores(ids, source_id, mode)
]
scores = pd.DataFrame(score_rows)

score_summary = (
    scores.groupby(["mode", "length"])
    .agg(
        samples=("z", "size"),
        raw_mean=("weighted_mean", "mean"),
        raw_std=("weighted_mean", "std"),
        z_mean=("z", "mean"),
        z_std=("z", "std"),
    )
    .reset_index()
)
display(score_summary.round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
colors = {"original": "tab:blue", "unwatermarked": "tab:orange", "watermarked": "tab:green"}
for axis, length in zip(axes.flat, PREFIXES):
    part = scores[scores["length"] == length]
    x = np.linspace(min(-4, part["z"].min() - 1), max(4, part["z"].max() + 1), 300)
    for mode, color in colors.items():
        values = part.loc[part["mode"] == mode, "z"]
        if len(values) > 1 and values.std() > 0:
            axis.plot(x, gaussian_kde(values)(x), color=color, linewidth=1.5, label=mode)
    axis.plot(x, norm.pdf(x), "k:", label="N(0, 1)")
    axis.axvline(
        THEORETICAL_Z_THRESHOLD, color="red", linestyle="--", label="1% FPR threshold"
    )
    axis.set(title=f"{length} scored contexts", xlabel="weighted-mean z-score", ylabel="density")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=5)
plt.show()





In [ ]:
# original 与 unwatermarked 为负类，watermarked 为正类。
evaluation = []
for length, part in scores.groupby("length"):
    positive = part["mode"] == "watermarked"
    predicted = part["z"] > THEORETICAL_Z_THRESHOLD
    evaluation.append({
        "length": length,
        "neg_samples": int((~positive).sum()),
        "pos_samples": int(positive.sum()),
        "raw_threshold": detector.raw_threshold(int(length), THEORETICAL_Z_THRESHOLD),
        "ROC-AUC": roc_auc_score(positive, part["weighted_mean"]),
        "Precision": precision_score(positive, predicted, zero_division=0),
        "Recall": recall_score(positive, predicted),
        "FPR": predicted[~positive].mean(),
    })

evaluation = pd.DataFrame(evaluation)
display(evaluation.round(3))





## 10、结论与限制

- SynthID-Text 的核心不是给固定“绿名单”加常数，而是用多层 Tournament 对完整概率分布做归一化更新；
  二选一配置在随机密钥期望下保持原分布。
- Weighted Mean 不需要训练，也不需要访问语言模型；它只依赖 tokenizer、密钥、哈希配置和 token 序列。
- raw score 的零假设中心是 0.5，但方差随有效文本长度变化。本 notebook 用理论方差转成 z-score，并同时报告
  实际 FPR。生产系统应按预期文本长度和真实负样本重新校准阈值。
- 重复 context 必须在嵌入端跳过、检测端掩码；否则重复短语会被重复计证，破坏非失真保证和显著性校准。
- 极简代码仍是 $O(BVm)$ 的全词表算法；代码行数减少不代表运行开销减少。只处理 top-k 候选可以更快，
  但必须固定截断顺序，并重新确认与目标生成实现的兼容性。
- 公开配置不能检测 Gemini 的线上水印；密钥、tokenizer 或哈希配置任一不匹配，检测结果都没有意义。
- 大幅改写、翻译或截取短片段会削弱任何生成式文字水印，因此水印检测只能作为来源证据之一。
